In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
load_dotenv()

class subgraph_state(TypedDict):
    input_text:str
    translate_text:str
    
subgraph_llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
def translate_text(state:subgraph_state):
    prompt=f"""translate the text in hindi keep it simple and natural donot add extra content
    text={state['input_text']}""".strip()
    translate_text=subgraph_llm.invoke(prompt).content
    return {'translate_text':translate_text}
graph=StateGraph(subgraph_state)
graph.add_node('translate_text',translate_text)
graph.add_edge(START,'translate_text')
graph.add_edge('translate_text',END)

subgraph=graph.compile()

class parent_state(TypedDict):
    question:str
    answer_eng:str
    answer_hindi:str
    
    
parent_llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

def generate_ans(state: parent_state):
    answer=f"""you are a helpful assistant answer clearly 
    question:{state['question']}""".content
    return {'answer_eng':answer}
def translate_ans(state:parent_state):
    result=subgraph.invoke({"input_text":state['answer_eng']})
    return {'answer_hindi':result['generate_ans']}

builder=StateGraph(parent_state)
builder.add_node('generate_ans',generate_ans)
builder.add_node('translate_ans',translate_ans)
builder.add_edge(START,'generate_ans')
builder.add_edge('generate_ans',translate_ans)
builder.add_edge('translate_ans',END)

res=builder.compile()
result=res.invoke({'question':"what is llm"})
print("English Answer:")
print(result["answer_eng"])

print("\nHindi Answer:")
print(result["answer_hindi"])

from IPython.display import Image
display(Image(res.get_graph().draw_mermaid_png()))






ValueError: Found edge ending at unknown node `<function translate_ans at 0x0000015C492468E0>`